In [1]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

import joblib

In [2]:
df = pd.read_csv("pakistan_SMS_spam_dataset.csv")
df.head()

,label,message
0,ham,Let's meet at the cafe after university.
1,ham,Aj match dekhne aa rahe ho?
2,ham,Your WhatsApp verification code is 959745.
3,ham,Where are you? We are waiting outside.
4,ham,Can you send me the notes for today's lecture?


In [3]:
df.info()
df['label'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    1000 non-null   object
 1   message  1000 non-null   object
dtypes: object(2)
memory usage: 15.8+ KB


label
ham     500
spam    500
Name: count, dtype: int64

In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = text.strip()
    return text

In [5]:
df['clean_text'] = df['message'].apply(clean_text)

In [6]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [7]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=3000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [9]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)

MultinomialNB()

In [10]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       112
           1       1.00      1.00      1.00        88

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [11]:
def predict_message(msg):
    cleaned = clean_text(msg)
    vector = vectorizer.transform([cleaned])
    prediction = model.predict(vector)

    if prediction[0] == 1:
        return "SPAM 🚨"
    else:
        return "HAM ✅"

In [12]:
print(predict_message("Congratulations! You won a free iPhone. Click here to claim now"))
print(predict_message("Hey, are we meeting tomorrow at university?"))
print(predict_message("URGENT! Your bank account is locked. Send OTP immediately"))
print(predict_message("Don’t forget to bring notes to class"))

SPAM 🚨
HAM ✅
SPAM 🚨
HAM ✅


In [13]:
print(predict_message("Tum ne lucky draw jeet liya hai, yahan click karo"))
print(predict_message("Kal assignment submit karna hai yaad rakhna"))
print(predict_message("Free prize mil raha hai jaldi reply karo"))

SPAM 🚨
HAM ✅
SPAM 🚨


In [14]:
test_msgs = [
    "Win a lottery now click link",
    "Meeting at 3 pm in lab",
    "Apka account suspend ho gaya hai verify karo",
    "Let's grab lunch today",
    "Free recharge offer limited time"
]

for msg in test_msgs:
    print(msg, "=>", predict_message(msg))

Win a lottery now click link => SPAM 🚨
Meeting at 3 pm in lab => HAM ✅
Apka account suspend ho gaya hai verify karo => SPAM 🚨
Let's grab lunch today => HAM ✅
Free recharge offer limited time => SPAM 🚨


In [15]:
joblib.dump(model, "spam_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

['vectorizer.pkl']

In [16]:
joblib.dump(model, "spam_model.pkl")


['spam_model.pkl']